In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler
import warnings
warnings.filterwarnings('ignore')


In [ ]:
class FinancialFeatureEngineer:
    """
    Comprehensive feature engineering for financial price prediction
    Binary classification: BUY if return > 0, SELL otherwise
    """
    
    def __init__(self, offset_days=20):
        self.offset_days = offset_days
        self.fitted_volatility_threshold = None
        self.scaler = None
        self.scaler_feature_names = None
        self.feature_columns = []

        
    def calculate_returns_and_moving_averages(self, df):
        """Calculate returns and moving averages"""
        df = df.copy()
        
        # Basic returns
        df['1d_return'] = df.groupby('Symbol')['Close'].pct_change()
        df['5d_ROC'] = df.groupby('Symbol')['Close'].pct_change(5)
        df['10d_ROC'] = df.groupby('Symbol')['Close'].pct_change(10)
        df['20d_ROC'] = df.groupby('Symbol')['Close'].pct_change(20)
        df['momentum_ratio'] = df.groupby('Symbol')['Close'].transform(lambda x: x / x.shift(5))
        
        # Simple Moving Averages
        df['SMA_10'] = df.groupby('Symbol')['Close'].transform(lambda x: x.rolling(10).mean())
        df['SMA_20'] = df.groupby('Symbol')['Close'].transform(lambda x: x.rolling(20).mean())
        df['SMA_50'] = df.groupby('Symbol')['Close'].transform(lambda x: x.rolling(50).mean())
        
        # Exponential Moving Averages
        df['EMA_12'] = df.groupby('Symbol')['Close'].transform(lambda x: x.ewm(span=12).mean())
        df['EMA_26'] = df.groupby('Symbol')['Close'].transform(lambda x: x.ewm(span=26).mean())
        df['ema_5'] = df.groupby('Symbol')['Close'].transform(lambda x: x.ewm(span=5).mean())
        df['ema_20'] = df.groupby('Symbol')['Close'].transform(lambda x: x.ewm(span=20).mean())
        
        # Price to MA ratios
        df['close_to_SMA_10'] = df['Close'] / df['SMA_10']
        df['close_to_SMA_20'] = df['Close'] / df['SMA_20']
        df['close_to_SMA_50'] = df['Close'] / df['SMA_50']
        df['ema_ratio'] = df['ema_5'] / df['ema_20']
        
        return df

    
    def calculate_momentum_indicators(self, df):
        """Calculate comprehensive momentum indicators"""
        # MACD
        df['MACD'] = df['EMA_12'] - df['EMA_26']
        df['MACD_signal'] = df.groupby('Symbol')['MACD'].transform(lambda x: x.ewm(span=9).mean())
        df['MACD_histogram'] = df['MACD'] - df['MACD_signal']
        
        # RSI
        def calculate_rsi(series, window=14):
            delta = series.diff()
            gain = delta.where(delta > 0, 0)
            loss = -delta.where(delta < 0, 0)
            
            avg_gain = gain.rolling(window=window).mean()
            avg_loss = loss.rolling(window=window).mean()
            
            rs = avg_gain / avg_loss
            rsi = 100 - (100 / (1 + rs))
            return rsi
        
        df['RSI_14'] = df.groupby('Symbol')['Close'].transform(calculate_rsi)
        
        def compute_williams_r(df):
            if df['Symbol'].nunique() == 1:
                high14 = df['High'].rolling(14).max()
                low14 = df['Low'].rolling(14).min()
                wr = (high14 - df['Close']) / (high14 - low14) * -100
                return wr

            return df.groupby('Symbol', group_keys=False).apply(
                lambda g: (g['High'].rolling(14).max() - g['Close']) /
                          (g['High'].rolling(14).max() - g['Low'].rolling(14).min()) * -100
            ).reset_index(level=0, drop=True)

        df['williams_R_14'] = compute_williams_r(df)
        
        return df
    
    def calculate_volatility_features(self, df):
        """Calculate comprehensive volatility features"""
        # Volatility measures
        df['volatility_10'] = df.groupby('Symbol')['1d_return'].transform(lambda x: x.rolling(10).std())
        df['volatility_20'] = df.groupby('Symbol')['1d_return'].transform(lambda x: x.rolling(20).std())
        df['volatility_60'] = df.groupby('Symbol')['1d_return'].transform(lambda x: x.rolling(60).std())
        
        # Normalized range
        df['normalized_range'] = (df['High'] - df['Low']) / df['Close']
        df['rolling_std_5'] = df.groupby('Symbol')['Close'].transform(lambda x: x.rolling(5).std())
        df['rolling_std_20'] = df.groupby('Symbol')['Close'].transform(lambda x: x.rolling(20).std())
        df['volatility_ratio'] = df['rolling_std_5'] / df['rolling_std_20']
        
        # Average True Range (ATR)
        def calculate_atr(group):
            high_low = group['High'] - group['Low']
            high_close_prev = abs(group['High'] - group['Close'].shift(1))
            low_close_prev = abs(group['Low'] - group['Close'].shift(1))
            true_range = pd.concat([high_low, high_close_prev, low_close_prev], axis=1).max(axis=1)
            return true_range.rolling(14).mean()
        
        df['ATR_14'] = df.groupby('Symbol').apply(calculate_atr).reset_index(level=0, drop=True)
        df['true_range'] = df['High'] - df['Low']
        df['atr_14'] = df.groupby('Symbol')['true_range'].transform(lambda x: x.rolling(14).mean())
        
        # Bollinger Bands
        df['bollinger_upper'] = df['SMA_20'] + 2 * df['volatility_20']
        df['bollinger_lower'] = df['SMA_20'] - 2 * df['volatility_20']
        df['bollinger_position'] = (df['Close'] - df['bollinger_lower']) / (df['bollinger_upper'] - df['bollinger_lower'])
        
        # Sharpe ratio
        df['sharpe_20'] = df.groupby('Symbol')['1d_return'].transform(
            lambda x: x.rolling(20).mean() / x.rolling(20).std()
        )
        
        return df
    
    def calculate_volume_features(self, df):
        """Calculate comprehensive volume features"""
        # Volume features
        df['volume_SMA_20'] = df.groupby('Symbol')['Volume'].transform(lambda x: x.rolling(20).mean())
        df['volume_SMA_10'] = df.groupby('Symbol')['Volume'].transform(lambda x: x.rolling(10).mean())
        df['volume_SMA_5'] = df.groupby('Symbol')['Volume'].transform(lambda x: x.rolling(5).mean())
        
        df['volume_ratio'] = df['Volume'] / df['volume_SMA_20']
        df['vol_ma_5'] = df.groupby('Symbol')['Volume'].transform(lambda x: x.rolling(5).mean())
        df['vol_ma_20'] = df.groupby('Symbol')['Volume'].transform(lambda x: x.rolling(20).mean())
        df['vol_ratio'] = df['vol_ma_5'] / df['vol_ma_20']
        
        # Check if VWAP exists before using it
        if 'VWAP' in df.columns:
            df['vwap_ratio'] = df['Close'] / df['VWAP']
        
        # On-Balance Volume (OBV)
        def calculate_obv(group):
            obv = (group['Volume'] * np.where(group['Close'] > group['Close'].shift(1), 1, 
                                           np.where(group['Close'] < group['Close'].shift(1), -1, 0))).cumsum()
            return obv
        
        df['OBV'] = df.groupby('Symbol').apply(calculate_obv).reset_index(level=0, drop=True)
        
        # Volume-return correlation
        def volume_return_corr(group):
            return group['1d_return'].rolling(20).corr(group['Volume'])
        
        df['volume_return_corr_20'] = df.groupby('Symbol').apply(volume_return_corr).reset_index(level=0, drop=True)
        
        return df
    
    def calculate_candlestick_features(self, df):
        """Calculate candlestick patterns and features"""
        df['candle_body'] = abs(df['Close'] - df['Open'])
        df['upper_shadow'] = df['High'] - df[['Close', 'Open']].max(axis=1)
        df['lower_shadow'] = df[['Close', 'Open']].min(axis=1) - df['Low']
        
        # Bullish engulfing pattern
        df['bullish_engulfing'] = ((df['Close'] > df['Open']) &
                                   (df['Close'].shift(1) < df['Open'].shift(1)) &
                                   (df['Close'] > df['Open'].shift(1)) &
                                   (df['Open'] < df['Close'].shift(1))).astype(int)
        
        return df
    
    def create_time_features(self, df):
        """Create comprehensive time-based features"""
        df['Date'] = pd.to_datetime(df['Date'])
        df['day_of_week'] = df['Date'].dt.dayofweek
        df['month'] = df['Date'].dt.month
        df['day_of_month'] = df['Date'].dt.day
        df['is_month_start'] = df['Date'].dt.is_month_start.astype(int)
        df['is_month_end'] = df['Date'].dt.is_month_end.astype(int)
        df['is_quarter_end'] = (df['Date'].dt.month % 3 == 0) & df['Date'].dt.is_month_end
        
        # One-hot encode categorical time features
        time_dummies = pd.get_dummies(df['day_of_week'], prefix='dow')
        month_dummies = pd.get_dummies(df['month'], prefix='month')
        
        df = pd.concat([df, time_dummies, month_dummies], axis=1)
        
        return df
    
    def create_tier1_raw_lags(self, df):
        """Create Tier 1: Raw lag features for critical indicators"""
        lag_features = {
            '1d_return': [1, 2, 3, 5],
            'volume_ratio': [1, 2, 5],
            'RSI_14': [1, 3, 5],
            'normalized_range': [1, 3],
            'vol_ratio': [1, 2, 3],
            'MACD': [1, 2],
            'bollinger_position': [1, 2]
        }
        
        if 'vwap_ratio' in df.columns:
            lag_features['vwap_ratio'] = [1, 2, 3]
            
        if 'ema_ratio' in df.columns:
            lag_features['ema_ratio'] = [1, 2, 3]

        
        for feature, lags in lag_features.items():
            if feature in df.columns:
                for lag in lags:
                    df[f'{feature}_lag_{lag}'] = df.groupby('Symbol')[feature].shift(lag)
        
        return df
    
    def create_tier2_engineered_aggregates(self, df):
        """Create Tier 2: Engineered lag aggregates"""
        base_series = ['normalized_range', 'bollinger_position', 
                      'volume_ratio', '1d_return', 'vol_ratio', 'RSI_14']
        
        if 'vwap_ratio' in df.columns:
            base_series.append('vwap_ratio')
            
        windows = [5, 10, 20]
        
        for series in base_series:
            if series in df.columns:  # Only if column exists
                for window in windows:
                    df[f'{series}_rolling_mean_{window}'] = df.groupby('Symbol')[series].transform(
                        lambda x: x.rolling(window).mean()
                    )
                    df[f'{series}_rolling_std_{window}'] = df.groupby('Symbol')[series].transform(
                        lambda x: x.rolling(window).std()
                    )
                    df[f'{series}_rolling_max_{window}'] = df.groupby('Symbol')[series].transform(
                        lambda x: x.rolling(window).max()
                    )
                    df[f'{series}_rolling_min_{window}'] = df.groupby('Symbol')[series].transform(
                        lambda x: x.rolling(window).min()
                    )
                    
                    # Rolling trend (slope of linear regression)
                    def rolling_trend(series, window):
                        if len(series) < window:
                            return np.nan
                        x = np.arange(window)
                        y = series.values[-window:]
                        if np.isnan(y).any(): # Handle NaNs in window
                            return np.nan
                        return np.polyfit(x, y, 1)[0]  # Return slope
                    
                    df[f'{series}_rolling_trend_{window}'] = df.groupby('Symbol')[series].transform(
                        lambda x: x.rolling(window).apply(lambda y: rolling_trend(y, window), raw=False)
                    )
        
        return df
    
    def create_tier3_event_memory(self, df):
        """Create Tier 3: Event and regime memory features"""
        # Max drawdown and runup
        df['rolling_max_20'] = df.groupby('Symbol')['Close'].transform(lambda x: x.rolling(20).max())
        df['rolling_min_20'] = df.groupby('Symbol')['Close'].transform(lambda x: x.rolling(20).min())
        
        df['max_drawdown_20'] = np.minimum(0, (df['Close'] / df['rolling_max_20']) - 1)
        df['max_runup_20'] = np.maximum(0, (df['Close'] / df['rolling_min_20']) - 1)
        df['price_position_20d'] = (df['Close'] - df['rolling_min_20']) / (df['rolling_max_20'] - df['rolling_min_20'])
        
        # Volatility regime flags
        if 'volatility_20' in df.columns:
            if self.fitted_volatility_threshold is None:
                # Calculate threshold only on non-NaN values
                self.fitted_volatility_threshold = df['volatility_20'].dropna().quantile(0.75)
            
            df['high_volatility_flag'] = (df['volatility_20'] > self.fitted_volatility_threshold).astype(int)
            df['high_volatility_flag_lag_5'] = df.groupby('Symbol')['high_volatility_flag'].shift(5)
        
        # Volume spike flag
        if 'volume_ratio' in df.columns:
            df['volume_spike_flag'] = (df['volume_ratio'] > 2.0).astype(int)
        
        # Large move flag
        if '1d_return' in df.columns and 'volatility_20' in df.columns:
            df['large_move_flag'] = (abs(df['1d_return']) > 2 * df['volatility_20']).astype(int)
        
        return df
    
    def create_target_variables(self, df):
        """Create binary target variables: BUY if return > 0, SELL otherwise"""
        # Calculate forward returns
        df['20d_fwd_return'] = (
            df.groupby('Symbol')['Close'].shift(-self.offset_days) / df['Close'] - 1
        )
        
        df['60d_fwd_return'] = (
            df.groupby('Symbol')['Close'].shift(-60) / df['Close'] - 1
        )
        
        # Binary classification: BUY if return > 0, SELL otherwise
        df['20d_signal'] = np.where(df['20d_fwd_return'] > 0, 'BUY', 'SELL')
        df['60d_signal'] = np.where(df['60d_fwd_return'] > 0, 'BUY', 'SELL')
        
        # Also create binary numeric targets for ML (1 for BUY, 0 for SELL)
        df['20d_target'] = (df['20d_fwd_return'] > 0).astype(int)
        df['60d_target'] = (df['60d_fwd_return'] > 0).astype(int)
        
        return df
    
    def clean_dataframe(self, df):
        """Remove unnecessary columns and clean the dataframe"""
        # Remove deliverable-related columns if they exist
        columns_to_remove = ['%Deliverble', 'Deliverable Volume','Series']
        
        # Only remove columns that exist in the dataframe
        columns_to_remove = [col for col in columns_to_remove if col in df.columns]
        df = df.drop(columns=columns_to_remove)
        
        return df
    
    def create_raw_features(self, df):
        """Create raw copies of key features before normalization"""
        df = df.copy()
        
        key_features = [
            'Close', 'Open', 'High', 'Low', 'VWAP', 'Volume',
            '1d_return', '20d_fwd_return', '60d_fwd_return',
            'ATR_14', 'true_range', 'volatility_10', 'volatility_20', 'volatility_60',
            'rolling_max_20', 'rolling_min_20', 'max_drawdown_20', 'max_runup_20'
        ]
        
        for col in key_features:
            if col in df.columns and f'raw_{col}' not in df.columns:
                df[f'raw_{col}'] = df[col].copy()
        
        return df
    
    def normalize_features(self, df, fit=False):
        """
        Robust normalization that PRESERVES raw price/return columns needed
        for backtesting/metrics. It creates `raw_<col>` copies for a safe list
        and scales only the remaining numeric features.
        """
        df = df.copy()

        # 1) Ensure Close exists
        if 'Close' not in df.columns:
            raise ValueError("df must contain a 'Close' column before normalization")

        # 2) Columns we explicitly want to preserve raw versions of (if present)
        preserve_candidates = [
            'Close', 'Open', 'High', 'Low', 'VWAP', 'Volume',
            '1d_return', '20d_fwd_return', '60d_fwd_return',
            'ATR_14', 'true_range', 'volatility_10', 'volatility_20', 'volatility_60',
            'rolling_max_20', 'rolling_min_20', 'max_drawdown_20', 'max_runup_20'
        ]

        # Create raw_ copies (only for columns present)
        for col in preserve_candidates:
            if col in df.columns and f'raw_{col}' not in df.columns:
                df[f'raw_{col}'] = df[col].copy()

        # Always keep raw_Close convenience column
        if 'raw_Close' not in df.columns:
            df['raw_Close'] = df['Close'].copy()

        # 3) Define targets and meta cols to never scale
        target_cols = ['20d_fwd_return', '60d_fwd_return', '20d_signal', '60d_signal', '20d_target', '60d_target']
        meta_cols = ['Date', 'Symbol']

        # Build set of columns we DO NOT want to scale
        do_not_scale = set(target_cols + meta_cols)
        # also add any raw_ columns we created
        do_not_scale.update([c for c in df.columns if c.startswith('raw_')])
        # and also keep original price columns unscaled (so we don't accidentally change 'Close')
        do_not_scale.update([c for c in preserve_candidates if c in df.columns])

        # 4) Numeric columns to consider for scaling: all numeric cols minus do_not_scale
        numeric_cols_all = df.select_dtypes(include=[np.number]).columns.tolist()
        cols_to_scale = [c for c in numeric_cols_all if c not in do_not_scale]

        if not cols_to_scale:
            # nothing to scale — preserve raw_Close and return
            return df

        # 5) Fit or transform
        if fit or self.scaler is None:
            self.scaler = RobustScaler()
            # Save the exact column order used for fit so we can transform consistently later
            self.scaler_feature_names = cols_to_scale.copy()
            df[self.scaler_feature_names] = self.scaler.fit_transform(df[self.scaler_feature_names])
        else:
            # Ensure the df contains all expected scaler feature names; add missing as zeros
            expected = list(self.scaler_feature_names)
            missing = [c for c in expected if c not in df.columns]
            if missing:
                for c in missing:
                    df[c] = 0.0
            # Transform in the same order
            df[expected] = self.scaler.transform(df[expected])

        return df
    
    def fit_transform(self, df, normalize=True):
        """Main method to apply all feature engineering and fit the scaler"""
        print("Starting comprehensive feature engineering...")
        
        # **CORRECTED ORDER:** Calculate features FIRST
        
        # Step 1: Returns and Moving Averages
        df = self.calculate_returns_and_moving_averages(df)
        print("✓ Returns and moving averages calculated")
        
        # Step 2: Momentum indicators
        df = self.calculate_momentum_indicators(df)
        print("✓ Momentum indicators calculated")
        
        # Step 3: Volatility features
        df = self.calculate_volatility_features(df)
        print("✓ Volatility features calculated")
        
        # Step 4: Volume features
        df = self.calculate_volume_features(df)
        print("✓ Volume features calculated")
        
        # Step 5: Candlestick features
        df = self.calculate_candlestick_features(df)
        print("✓ Candlestick features calculated")
        
        # Step 6: Time features
        df = self.create_time_features(df)
        print("✓ Time features created")
        
        # Step 7: Tier 1 - Raw lags
        df = self.create_tier1_raw_lags(df)
        print("✓ Tier 1 raw lags created")
        
        # Step 8: Tier 2 - Engineered aggregates
        df = self.create_tier2_engineered_aggregates(df)
        print("✓ Tier 2 engineered aggregates created")
        
        # Step 9: Tier 3 - Event memory
        df = self.create_tier3_event_memory(df)
        print("✓ Tier 3 event memory features created")
        
        # Step 10: Target variables
        df = self.create_target_variables(df)
        print("✓ Target variables created")
        
        # Step 11: Clean dataframe
        df = self.clean_dataframe(df)
        print("✓ Dataframe cleaned")
        
        # Step 12: Normalize features AFTER they are created
        if normalize:
            df = self.normalize_features(df, fit=True)
            print("✓ Features normalized using RobustScaler")
        
        # Remove rows with NaN values (from rolling calculations)
        original_shape = df.shape[0]
        df = df.dropna()
        print(f"✓ Removed rows with NaN. Final dataset: {df.shape[0]} rows (from {original_shape})")
        
        # Store feature column names
        target_cols = ['20d_fwd_return', '60d_fwd_return', '20d_signal', '60d_signal', '20d_target', '60d_target', 'raw_20d_fwd_return', 'raw_60d_fwd_return']
        meta_cols = ['Date', 'Symbol','raw_Close']
        self.feature_columns = [col for col in df.columns if col not in target_cols and col not in meta_cols]
        
        print(f"✓ Feature engineering complete! Total features: {len(self.feature_columns)}")
        
        # Print target distribution
        if '20d_signal' in df.columns:
            print(f"\nTarget distribution (20-day):")
            print(df['20d_signal'].value_counts(normalize=True))
        
        return df

    def transform(self, df, normalize=True):
        """Apply feature engineering using the fitted scaler"""
        print("Starting feature transformation...")
        
        # Apply all feature creation steps
        df = self.calculate_returns_and_moving_averages(df)
        df = self.calculate_momentum_indicators(df)
        df = self.calculate_volatility_features(df)
        df = self.calculate_volume_features(df)
        df = self.calculate_candlestick_features(df)
        df = self.create_time_features(df)
        df = self.create_tier1_raw_lags(df)
        df = self.create_tier2_engineered_aggregates(df)
        df = self.create_tier3_event_memory(df)
    
        
        # Create targets (they will be mostly NaN for future data, which is fine)
        df = self.create_target_variables(df)
        
        # Clean dataframe
        df = self.clean_dataframe(df)
        
        # Normalize using the ALREADY FITTED scaler
        if normalize:
            if self.scaler is None:
                print("Warning: Scaler not fitted. Fitting on current data.")
                df = self.normalize_features(df, fit=True)
            else:
                df = self.normalize_features(df, fit=False)
                print("✓ Features normalized using fitted RobustScaler")

        # Do NOT drop NaNs here, as the model will predict on the last row
        df = df.fillna(0)
        
        
        if 'raw_Close' not in df.columns:
            df['raw_Close'] = df['Close'].copy()
            
        # Update feature columns list based on what actually exists
        target_cols = ['20d_fwd_return', '60d_fwd_return', '20d_signal', '60d_signal', '20d_target', '60d_target','raw_Close']
        meta_cols = ['Date', 'Symbol','raw_Close', 'raw_Open', 'raw_High', 'raw_Low', 'raw_VWAP', 'raw_Volume', 'pred'] 
        
        # Dynamically filter columns that actually exist in df
        self.feature_columns = [col for col in df.columns if col not in target_cols and col not in meta_cols]

        print(f"✓ Feature transformation complete!")
        
        return df
    
    def get_feature_categories(self, df):
        """Analyze and return feature categories"""
        feature_categories = {
            'price_momentum': [col for col in df.columns if any(x in col for x in [
                'return', 'ROC', 'SMA', 'EMA', 'MACD', 'RSI', 'close_to', 'momentum', 'williams'
            ])],
            'volatility': [col for col in df.columns if any(x in col for x in [
                'volatility', 'range', 'ATR', 'bollinger', 'sharpe', 'std', 'atr'
            ])],
            'volume': [col for col in df.columns if any(x in col for x in [
                'volume', 'OBV', 'vol_', 'vwap'
            ])],
            'candlestick': [col for col in df.columns if any(x in col for x in [
                'candle', 'shadow', 'engulfing'
            ])],
            'time': [col for col in df.columns if any(x in col for x in [
                'dow', 'month', 'is_'
            ])],
            'lags_tier1': [col for col in df.columns if '_lag_' in col],
            'aggregates_tier2': [col for col in df.columns if any(x in col for x in [
                'rolling_mean', 'rolling_std', 'rolling_max', 'rolling_min', 'rolling_trend'
            ])],
            'memory_tier3': [col for col in df.columns if any(x in col for x in [
                'drawdown', 'runup', 'position', 'flag'
            ])]}
        
        # Print feature counts
        total_features = 0
        for category, features in feature_categories.items():
            print(f"{category}: {len(features)} features")
            total_features += len(features)
        
        print(f"\nTotal features across categories: {total_features}")
        
        return feature_categories

In [ ]:
def process_unseen_data(file_path):
    df_unseen = pd.read_csv(file_path)

    df_unseen = df_unseen.drop(['52W H ', '52W L '], axis=1)
    rename_map = {
        'Date ': 'Date',
        'series ': 'Series',
        'OPEN ': 'Open',
        'HIGH ': 'High',
        'LOW ': 'Low',
        'PREV. CLOSE ': 'Prev Close',
        'ltp ': 'Last',
        'close ': 'Close',
        'vwap ': 'VWAP',
        'VOLUME ': 'Volume',
        'VALUE ': 'Turnover',
        'No of trades ': 'Trades',
        'raw_Close': 'raw_Close'
    }
    df_unseen = df_unseen.rename(columns=rename_map)
    df_unseen['Date'] = pd.to_datetime(df_unseen['Date'])
    df_unseen = df_unseen.sort_values(['Date']).reset_index(drop=True)

    cols_to_float = ['Prev Close', 'Open', 'High', 'Low', 'Last', 'Close', 'VWAP', 'Volume', 'Turnover', 'Trades']

    for col in cols_to_float:
        df_unseen[col] = (
        df_unseen[col]
        .astype(str)                      # ensure string for replacements
        .str.replace(',', '', regex=False) # remove commas
        .str.replace('₹', '', regex=False) # remove currency symbols
        .str.replace('%', '', regex=False) # remove percentage signs if any
        .str.strip()                       # remove leading/trailing spaces
        )
        df_unseen[col] = pd.to_numeric(df_unseen[col], errors='coerce')
        
    df_unseen['Symbol'] = 'TEST'
    df_unseen.loc[0, 'Symbol'] = 'TEST01'

    return df_unseen

In [ ]:
def fill_missing_values(df,feature_columns):
    for col in feature_columns:
        if col not in df.columns:
            print(f"Adding missing feature column: {col}")
            df[col] = 0
    
    return df